In [3]:

import asyncio
import warnings
import pandas as pd
from pathlib import Path

# Парсеры
from antimony import antimony_parser
from westmetall import westmetall_async
from lme import lme_selenium_async
from lbma import lbma_prescious_async
from cbr import cb_currency, cb_metalls
from nbk import nbk_tenge_async
from shmet import shmet_optimized_async
from kitco import kitco_parser_async

# Сервисные функции из service_layer
from service_layer import (
    read_db,
    save_db,
    check_df,
    check_and_save_pair,
    show_db,
    excel_to_csv_db
)

warnings.filterwarnings("ignore")


# ================================================================
# Пути к базам (реальная структура проекта)
# ================================================================
LME_PATH = Path("lme/data/LME_db_new.xlsx")
WESTMETALL_PATH = Path("westmetall/data/LME_westmetall_db.xlsx")

KITCO_PATH = Path("kitco/data/kitko_db.xlsx")
LBMA_PATH = Path("lbma/data/lbma_kitco_subs.xlsx")

ANTIMONY_PATH = Path("antimony/data/antimony.xlsx")

CB_CURRENCY_PATH = Path("cbr/data/cb_currency.xlsx")
CB_METALLS_PATH = Path("cbr/data/cb_metalls.xlsx")

NBK_PATH = Path("nbk/data/nbk_tenge.xlsx")
SHMET_PATH = Path("shmet/data/shmet_historical.xlsx")


# ================================================================
# Проверка целостности парных баз
# ================================================================
def db_check():
    """
    Проверка целостности парных баз с очисткой дубликатов
    и сохранением первого (более раннего) вхождения.
    """
    print("Проверка LME / Westmetall...")
    check_and_save_pair(
        LME_PATH,
        WESTMETALL_PATH,
        pair_name="LME / Westmetall",
        index=False,
    )

    print("Проверка Kitco / LBMA...")
    check_and_save_pair(
        KITCO_PATH,
        LBMA_PATH,
        pair_name="Kitco / LBMA",
        index=False,
    )
    
async def main():
    print("Parsing started...")

    tasks = {
        "lme": lme_selenium_async(),
        "antimony": antimony_parser(),
        "westmetall": westmetall_async(),
        "lbma": lbma_prescious_async(),
        "kitco": kitco_parser_async(),
        "cb_currency": cb_currency(),
        "cb_metalls": cb_metalls(),
        "nbk": nbk_tenge_async(),
        "shmet": shmet_optimized_async(),
    }

    # return_exceptions=True — чтобы падение одного парсера
    # не останавливало остальные
    results = await asyncio.gather(
        *tasks.values(),
        return_exceptions=True,
    )

    for name, result in zip(tasks.keys(), results):
        if isinstance(result, Exception):
            print(f"❌ Ошибка в {name}: {result}")

    print("All tasks are done!")
    print("+" * 64)
    print("Checking DB...")

    db_check()

    print("DB check completed!")
    print("+" * 64)
    print("Visual control")
    print("+" * 64)
    
    print("Converting Excel to CSV...")
    excel_to_csv_db()
    print("CSV conversion completed!")

    # Базовые металлы
    show_db("lme_selenium_db", LME_PATH, sheet_name=0)
    show_db("westmetall_db", WESTMETALL_PATH, sheet_name=0)

    # Драгоценные металлы
    show_db("kitco_db", KITCO_PATH, sheet_name=0)
    show_db("lbma_precious_db", LBMA_PATH, sheet_name=0)

    # Антимоний
    show_db("antimony_db", ANTIMONY_PATH, sheet_name=0)

    # ЦБ РФ: валюты (каждая на своем листе)
    for currency in [
        "USD",
        "EUR",
        "British_Pound",
        "China_Yuan",
        "Japanese_Yen",
        "Swiss_Franc",
    ]:
        show_db(
            f"cb_currency ({currency})",
            CB_CURRENCY_PATH,
            sheet_name=currency,
        )

    # ЦБ РФ: металлы
    show_db("cb_metalls_db", CB_METALLS_PATH, sheet_name=0)

    # Казахстан и SHMET
    show_db("nbk_tenge_db", NBK_PATH, sheet_name=0)
    show_db("shmet_historical_db", SHMET_PATH, sheet_name=0, show_head=True)


# Для Jupyter используем await, а не asyncio.run()
await main()

Parsing started...
🚀 LME parsing started...
antimony parsing is DONE
NBK_tenge parsing is DONE! (1675 строк)
✅ LME_main is done!!!
CB_metalls parsing is DONE!
USD is done!
EUR is done!
Australian_Dollar is done!
China_Yuan is done!
British_Pound is done!
Kazakhstan_Tenge is done!
Japanese_Yen is done!
Swiss_Franc is done!
CB_currency parsing is DONE!
WESTMETALL is done!!!
Произошла ошибка KITCO: Не найдены блоки <div class='grid'> на странице Kitco
LBMA is done!!!
SHMET is done!!!
All tasks are done!
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Checking DB...
Проверка LME / Westmetall...
LME / Westmetall: добавлены пропущенные даты и удалены дубликаты
Проверка Kitco / LBMA...
Kitco / LBMA: добавлены пропущенные даты и удалены дубликаты
DB check completed!
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Visual control
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Converting Excel to CSV...
Поиск Excel файлов...
Найдено 9 Excel файл

,date,aluminium,copper,lead,nickel,zink,tin
1177,2026-08-26,3227.5,14525.0,1866.0,16895,4107.0,55300
1178,2026-08-27,3212.5,14490.0,1872.0,16660,4107.0,54950
1179,2026-08-28,3222.0,14535.0,1880.0,16850,4070.0,55125
1180,2026-09-01,3261.0,14395.5,1874.0,16350,4115.0,54725
1181,2026-09-02,3255.5,14355.0,1873.0,16530,4000.0,53950


westmetall_db


,date,aluminium,copper,lead,nickel,zink,tin
1177,2026-08-26,3227.5,14525.0,1866.0,16895,4107.0,55300
1178,2026-08-27,3212.5,14490.0,1872.0,16660,4107.0,54950
1179,2026-08-28,3222.0,14535.0,1880.0,16850,4070.0,55125
1180,2026-09-01,3261.0,14395.5,1874.0,16350,4115.0,54725
1181,2026-09-02,3255.5,14355.0,1873.0,16530,4000.0,53950


kitco_db


...

,date,gold,silver,platinum,palladium
900,2026-08-28,12799.09,189.29,5131.52,3711.36
901,2026-08-29,12574.32,188.44,5012.44,3599.64
902,2026-09-01,12671.47,195.12,5232.02,4009.94
903,2026-09-02,12726.29,195.97,5254.66,4027.28
904,2026-09-03,12175.73,181.15,4944.53,3716.93


nbk_tenge_db


,date,Числовое значение,ДОЛЛАР США
1670,2026-08-30,1,464.77
1671,2026-08-31,1,464.77
1672,2026-09-01,1,462.31
1673,2026-09-02,1,458.45
1674,2026-09-03,1,455.40


shmet_historical_db


,date,price,unit
0,2020-01-10,48605,Yuan/MT
1,2020-01-14,48990,Yuan/MT
2,2020-01-15,49060,Yuan/MT
3,2020-01-16,48950,Yuan/MT
4,2020-01-17,48930,Yuan/MT
